# Capítulo 4: Variáveis Aleatórias Contínuas

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem. A seção 4.6 (exponencial) é leitura complementar e não entra aqui.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

Importa as bibliotecas usadas no capítulo. O `set_printoptions` faz o numpy mostrar números sem o prefixo `np.float64`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import quad
from formato import num

np.set_printoptions(legacy="1.25")

## 4.1 Variável Aleatória Contínua e Densidade

Simula 100.000 paradas do ponteiro do relógio e desenha histogramas de densidade com faixas de 30° e de 5°.

In [ ]:
rng = np.random.default_rng(42)
angulos = rng.uniform(0, 360, size=100_000)

fig, eixos = plt.subplots(1, 2, sharey=True, figsize=(7, 2.8))
for eixo, largura in zip(eixos, [30, 5]):
    eixo.hist(angulos, bins=np.arange(0, 361, largura), density=True)
    eixo.axhline(1/360, color="black", linestyle="--", linewidth=1)
    eixo.set_title(f"faixas de {largura}°")
    eixo.set_xlabel("ângulo x")
eixos[0].set_ylabel("densidade")
plt.tight_layout()
plt.show()

Proporção de ângulos simulados entre 0° e 90°, perto de 1/4.

In [ ]:
round(float(((angulos >= 0) & (angulos <= 90)).mean()), 4)

Probabilidades da densidade 2x em [0, 1] como áreas, calculadas com `quad`.

In [ ]:
f = lambda x: 2 * x
area_1, _ = quad(f, 0, 0.5)
area_2, _ = quad(f, 0.5, 1)
round(area_1, 4), round(area_2, 4)

## 4.2 Valor Médio e Variância

Aproxima o valor médio da densidade 2x por somas com faixas cada vez mais estreitas.

In [ ]:
def media_aproximada(n):
    h = 1 / n
    pontos_medios = (np.arange(n) + 0.5) * h
    return (pontos_medios * 2 * pontos_medios * h).sum()

pd.DataFrame({"faixas": [4, 10, 100, 1000],
              "aproximação": [media_aproximada(n) for n in [4, 10, 100, 1000]]}).round(6)

Valor médio e variância da densidade 2x e do relógio, por integração numérica.

In [ ]:
E_2x, _ = quad(lambda x: x * 2 * x, 0, 1)
E2_2x, _ = quad(lambda x: x**2 * 2 * x, 0, 1)

E_relogio, _ = quad(lambda x: x / 360, 0, 360)
Var_relogio, _ = quad(lambda x: (x - 180) ** 2 / 360, 0, 360)

print(round(E_2x, 4), round(E2_2x - E_2x**2, 4))
print(round(E_relogio, 2), round(Var_relogio, 2), round(np.sqrt(Var_relogio), 2))

Simula a densidade 2x com o gerador triangular do numpy e compara média e variância.

In [ ]:
rng = np.random.default_rng(42)
x = rng.triangular(left=0, mode=1, right=1, size=100_000)   # densidade 2x em [0, 1]
round(float(x.mean()), 4), round(float(x.var()), 4)

## 4.3 Função de Distribuição Acumulada

Densidade 2x e sua f.d.a. x², lado a lado.

In [ ]:
x = np.linspace(-0.2, 1.2, 400)
f = np.where((x >= 0) & (x <= 1), 2 * x, 0)
f[np.argmax(x > 1)] = np.nan          # corta o traço vertical no salto em x = 1
F = np.clip(x, 0, 1) ** 2

fig, (esq, dir) = plt.subplots(1, 2, figsize=(7, 2.8))
esq.plot(x, f)
esq.fill_between(x, f, where=(x >= 0) & (x <= 0.5), alpha=0.3)
esq.set_title("densidade f(x)")
dir.plot(x, F)
dir.plot(0.5, 0.25, "o")
dir.set_title("f.d.a. F(x)")
for eixo in (esq, dir):
    eixo.set_xlabel("x")
plt.tight_layout()
plt.show()

Probabilidade de um intervalo pela f.d.a. e pela integral da densidade.

In [ ]:
F_2x = lambda t: np.clip(t, 0, 1) ** 2

pela_fda = F_2x(0.8) - F_2x(0.3)
pela_integral, _ = quad(lambda t: 2 * t, 0.3, 0.8)
round(float(pela_fda), 4), round(pela_integral, 4)

## 4.4 Distribuição Uniforme Contínua

Densidade e f.d.a. da uniforme em [0, 60], o tempo de espera do jitter.

In [ ]:
jitter = stats.uniform(loc=0, scale=60)
x = np.linspace(-10, 70, 400)

fig, (esq, dir) = plt.subplots(1, 2, figsize=(7, 2.8))
esq.plot(x, jitter.pdf(x))
esq.set_title("densidade f(x)")
dir.plot(x, jitter.cdf(x))
dir.set_title("f.d.a. F(x)")
for eixo in (esq, dir):
    eixo.set_xlabel("espera x (s)")
plt.tight_layout()
plt.show()

Média, variância e probabilidades do jitter com `stats.uniform` (`scale` é a largura do intervalo).

In [ ]:
jitter = stats.uniform(loc=0, scale=60)          # u(0, 60)

print(jitter.mean(), jitter.var(), round(jitter.std(), 2))
print(jitter.cdf(15), round(jitter.cdf(45) - jitter.cdf(30), 2))

Uma uniforme qualquer a partir de números uniformes em [0, 1].

In [ ]:
rng = np.random.default_rng(42)
u = rng.random(100_000)
espera = 0 + (60 - 0) * u

round(float(espera.mean()), 2), round(float(espera.var()), 1), 60**2 / 12

## 4.5 Distribuição Normal

Três densidades normais: mudar a média desloca o sino, mudar o desvio-padrão o achata.

In [ ]:
x = np.linspace(-7, 8, 600)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
for mu, sigma in [(0, 1), (3, 1), (0, 2)]:
    ax.plot(x, stats.norm(loc=mu, scale=sigma).pdf(x), label=f"N({mu}, {sigma**2})")
ax.set_xlabel("x")
ax.set_ylabel("densidade")
ax.legend()
plt.show()

Probabilidades da normal padrão para z = 1,73, as mesmas da Tabela III do Bussab.

In [ ]:
Z = stats.norm(loc=0, scale=1)

print(round(Z.cdf(1.73) - 0.5, 4))           # P(0 ≤ Z ≤ 1,73), o valor da Tabela III
print(round(Z.sf(1.73), 4))                  # P(Z ≥ 1,73)
print(round(Z.cdf(-1.73), 4))                # P(Z ≤ -1,73), igual pela simetria
print(round(Z.cdf(1.73) - Z.cdf(0.47), 4))   # P(0,47 ≤ Z ≤ 1,73)

P(2 ≤ X ≤ 5) para X ~ N(3, 16) pela padronização e direto (`scale` é o desvio-padrão, 4).

In [ ]:
X = stats.norm(loc=3, scale=4)                 # N(3, 16): scale é o desvio-padrão

pela_padronizacao = Z.cdf(0.5) - Z.cdf(-0.25)
direto = X.cdf(5) - X.cdf(2)
round(pela_padronizacao, 4), round(direto, 4)

Probabilidade a até 1, 2 e 3 desvios-padrão da média.

In [ ]:
pd.DataFrame({
    "até k desvios": [1, 2, 3],
    "P(|Z| ≤ k)": [Z.cdf(k) - Z.cdf(-k) for k in (1, 2, 3)],
}).round(4)

Os depósitos do Exemplo 7.9 do Bussab.

In [ ]:
deposito = stats.norm(loc=10_000, scale=1_500)

print(deposito.cdf(10_000), deposito.sf(10_000))
print(round(deposito.cdf(15_000) - deposito.cdf(12_000), 4))
print(f"{deposito.sf(20_000):.1e}")